<a href="https://colab.research.google.com/github/zhouning/alphaearth-training-system/blob/paper12-results-colab-20260619/colab/paper12_eurosat_channel_bridge_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/zhouning/alphaearth-training-system/blob/paper12-results-colab-20260619/colab/paper12_eurosat_channel_bridge_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

                # Paper 12 EuroSAT Channel-Bridge Ablation

                This notebook runs the public EuroSAT ablation that compares deterministic zero padding against the learned 10->6 channel bridge for Prithvi adaptation.

                **Required runtime:** Colab Pro L4. A100 is optional. T4 is acceptable only for a short smoke run, not for the full 4-method x 3-seed x 50-epoch matrix.

                **Storage policy:** download EuroSAT to Colab local SSD under `/content/AlphaEarth-System/data/eurosat`, keep checkpoints under `/content/eurosat_channel_bridge_runs`, and only persist result JSON files to `/content/drive/MyDrive/paper12_results`.

                **Methods in this run:** `zero_pad_linear_probe`, `learned_bridge_linear_probe`, `zero_pad_houlsby`, `learned_bridge_houlsby`.

In [1]:
# 1. Mount Drive and create the results directory.
from google.colab import drive
drive.mount("/content/drive")

import os

RESULTS_DIR = "/content/drive/MyDrive/paper12_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Drive results directory:", RESULTS_DIR)

Mounted at /content/drive
Drive results directory: /content/drive/MyDrive/paper12_results


In [2]:
# 2. GPU, Python, and disk sanity check.
!nvidia-smi
!python --version
!df -h /content

Sat Jun 20 03:02:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   42C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 3. Clone the Paper 12 results branch into local SSD.
%cd /content
!rm -rf /content/AlphaEarth-System
!git clone --branch paper12-results-colab-20260619 https://github.com/zhouning/alphaearth-training-system.git /content/AlphaEarth-System
%cd /content/AlphaEarth-System
!git rev-parse --abbrev-ref HEAD
!git rev-parse HEAD
!git log --oneline -3

/content
Cloning into '/content/AlphaEarth-System'...
remote: Enumerating objects: 1196, done.
remote: Counting objects: 100% (360/360), done.
remote: Compressing objects: 100% (259/259), done.
remote: Total 1196 (delta 166), reused 271 (delta 96), pack-reused 836 (from 1)
Receiving objects: 100% (1196/1196), 54.83 MiB | 18.60 MiB/s, done.
Resolving deltas: 100% (548/548), done.
/content/AlphaEarth-System
paper12-results-colab-20260619
c79ba9c85fbe0d3fb68d60f006557eefa53b0ac5
c79ba9c (HEAD -> paper12-results-colab-20260619, origin/paper12-results-colab-20260619) fix: archive stale eurosat channel bridge outputs
ea95046 fix: pin paper12 colab notebooks to results branch
08dc6fc feat: add paper12 loveda r2u finetune results


In [4]:
# 4. Install the local package and notebook-only helpers.
%cd /content/AlphaEarth-System
!pip install -q -e . torchgeo pyyaml huggingface_hub

/content/AlphaEarth-System
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 871.8/871.8 kB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
# 5. Stage Prithvi weights at the path the benchmark expects.
%cd /content/AlphaEarth-System
import os
import shutil
from huggingface_hub import hf_hub_download

DRIVE_WEIGHTS = "/content/drive/MyDrive/Prithvi_100M.pt"
LOCAL_WEIGHTS = "/content/AlphaEarth-System/data/weights/prithvi/Prithvi_100M.pt"
os.makedirs(os.path.dirname(LOCAL_WEIGHTS), exist_ok=True)

if os.path.exists(DRIVE_WEIGHTS):
                    shutil.copy(DRIVE_WEIGHTS, LOCAL_WEIGHTS)
                    print("Copied Prithvi weights from Drive.")
elif not os.path.exists(LOCAL_WEIGHTS):
                    downloaded = hf_hub_download(
                        repo_id="ibm-nasa-geospatial/Prithvi-100M",
                        filename="Prithvi_100M.pt",
                    )
                    shutil.copy(downloaded, LOCAL_WEIGHTS)
                    print("Downloaded Prithvi weights from Hugging Face.")
else:
                    print("Prithvi weights already present locally.")

!ls -lh /content/AlphaEarth-System/data/weights/prithvi

/content/AlphaEarth-System
Copied Prithvi weights from Drive.
total 433M
-rw------- 1 root root 433M Jun 20 03:04 Prithvi_100M.pt


In [6]:
# 6. Download the public EuroSAT cache into local SSD and smoke one sample per split.
%cd /content/AlphaEarth-System
EUROSAT_ROOT = "/content/AlphaEarth-System/data/eurosat"
!python scripts/download_public_datasets.py --dataset eurosat --eurosat-root data/eurosat --max-samples 1
!du -sh /content/AlphaEarth-System/data/eurosat

/content/AlphaEarth-System
[ok] EuroSAT train sample=0 len=16200 image_shape=(10, 64, 64) label=0
[ok] EuroSAT test sample=0 len=5400 image_shape=(10, 64, 64) label=0
4.8G	/content/AlphaEarth-System/data/eurosat


In [7]:
# 7. Dry-run the full EuroSAT matrix before training.
%cd /content/AlphaEarth-System
!python -m geoadapter.bench.run_benchmark --config geoadapter/bench/configs/eurosat_channel_bridge.yaml --dry-run

/content/AlphaEarth-System
Total experiments: 12
  zero_pad_linear_probe x s2_full x seed=42
  zero_pad_linear_probe x s2_full x seed=123
  zero_pad_linear_probe x s2_full x seed=456
  learned_bridge_linear_probe x s2_full x seed=42
  learned_bridge_linear_probe x s2_full x seed=123
  learned_bridge_linear_probe x s2_full x seed=456
  zero_pad_houlsby x s2_full x seed=42
  zero_pad_houlsby x s2_full x seed=123
  zero_pad_houlsby x s2_full x seed=456
  learned_bridge_houlsby x s2_full x seed=42
  learned_bridge_houlsby x s2_full x seed=123
  learned_bridge_houlsby x s2_full x seed=456


In [8]:
# 8. Archive any pre-rerun EuroSAT JSON files so the benchmark cannot resume from archive output.
from datetime import datetime
import shutil
from pathlib import Path

results_dir = Path("/content/drive/MyDrive/paper12_results")
archive_dir = results_dir / "eurosat_channel_bridge_archive_pre_rerun"
archive_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for name in ["eurosat_channel_bridge.json", "eurosat_channel_bridge_summary.json"]:
                    src = results_dir / name
                    if src.exists():
                        dst = archive_dir / f"{stamp}_{name}"
                        shutil.move(str(src), str(dst))
                        print("Archived", src, "->", dst)
                    else:
                        print("No existing", src)

Archived /content/drive/MyDrive/paper12_results/eurosat_channel_bridge.json -> /content/drive/MyDrive/paper12_results/eurosat_channel_bridge_archive_pre_rerun/20260620_030617_eurosat_channel_bridge.json
Archived /content/drive/MyDrive/paper12_results/eurosat_channel_bridge_summary.json -> /content/drive/MyDrive/paper12_results/eurosat_channel_bridge_archive_pre_rerun/20260620_030617_eurosat_channel_bridge_summary.json


In [ ]:
# 9. Run the 4-method x 3-seed EuroSAT benchmark. Checkpoints stay on local SSD.
%cd /content/AlphaEarth-System
!mkdir -p /content/eurosat_channel_bridge_runs
!python -m geoadapter.bench.run_benchmark --config geoadapter/bench/configs/eurosat_channel_bridge.yaml --output /content/drive/MyDrive/paper12_results/eurosat_channel_bridge.json --checkpoint-dir /content/eurosat_channel_bridge_runs --checkpoint-every 5

/content/AlphaEarth-System
Total experiments: 12
  [zero_pad_linear_probe|s2_full|seed=42] device=cuda, trainable_params=7,690
  [zero_pad_linear_probe|s2_full|seed=42] Loaded eurosat: 16200 train, 5400 val
  [zero_pad_linear_probe|s2_full|seed=42] Epoch 10/50 loss=1.0006
  [zero_pad_linear_probe|s2_full|seed=42] Epoch 20/50 loss=0.9434
  [zero_pad_linear_probe|s2_full|seed=42] Epoch 30/50 loss=0.9206
  [zero_pad_linear_probe|s2_full|seed=42] Epoch 40/50 loss=0.9090
  [zero_pad_linear_probe|s2_full|seed=42] Epoch 50/50 loss=0.9059
  [zero_pad_linear_probe|s2_full|seed=42] OA=0.6894 F1=0.6752
  -> appended to /content/drive/MyDrive/paper12_results/eurosat_channel_bridge.json (1/12)
  [zero_pad_linear_probe|s2_full|seed=123] device=cuda, trainable_params=7,690
  [zero_pad_linear_probe|s2_full|seed=123] Loaded eurosat: 16200 train, 5400 val
  [zero_pad_linear_probe|s2_full|seed=123] Epoch 10/50 loss=1.0010
  [zero_pad_linear_probe|s2_full|seed=123] Epoch 20/50 loss=0.9396
  [zero_pad_line

In [ ]:
# 10. Verify result counts, aggregate OA and macro-F1 by method, and persist a compact summary JSON to Drive.
import json
from collections import defaultdict
from pathlib import Path
from statistics import mean, stdev

results_dir = Path("/content/drive/MyDrive/paper12_results")
results_path = results_dir / "eurosat_channel_bridge.json"
summary_path = results_dir / "eurosat_channel_bridge_summary.json"

rows = json.loads(results_path.read_text(encoding="utf-8"))
expected_rows = 12
assert len(rows) == expected_rows, f"expected {expected_rows} rows, got {len(rows)}"

grouped = defaultdict(list)
for row in rows:
                    grouped[row["method"]].append(row)

summary = {}
for method, method_rows in sorted(grouped.items()):
                    oa = [float(row["overall_accuracy"]) for row in method_rows]
                    f1 = [float(row["macro_f1"]) for row in method_rows]
                    summary[method] = {
                        "overall_accuracy_mean": mean(oa),
                        "overall_accuracy_std": stdev(oa) if len(oa) > 1 else 0.0,
                        "macro_f1_mean": mean(f1),
                        "macro_f1_std": stdev(f1) if len(f1) > 1 else 0.0,
                        "seeds": [int(row["seed"]) for row in method_rows],
                    }

for method, payload in summary.items():
                    print(method, payload)

summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Wrote", summary_path)